In [1]:
# Section 1
from dotenv import load_dotenv
import langchain
from langchain_ollama import ChatOllama

load_dotenv()
llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)

c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
from langchain_core.tools import tool

# Reusing the familiar tools from previous lessons
@tool
def get_weather(city: str) -> str:
    """Returns the current weather for a given city."""
    return f"The weather in {city} is sunny with a high of 28°C."

@tool
def get_stock_price(ticker: str) -> str:
    """Returns the current stock price for a given ticker symbol.
    Use 'ZENSAR' for Zensar Technologies, 'GOOGL' for Google."""
    prices = {"ZENSAR": "464.00 INR", "GOOGL": "175.00 USD"}
    return prices.get(ticker.upper(), f"Unknown ticker: {ticker}")

print("Tools ready: get_weather, get_stock_price")



Tools ready: get_weather, get_stock_price


In [3]:
# Approach 1 - Long term memory using InMemoryStore
from langgraph.store.memory import InMemoryStore

store = InMemoryStore() # works on user_id
namespace = ("memories", "user_id_tom")

# put(namespace, key, value)
store.put(namespace, "south_indian_receipes", {"text": "I like idli, wada, dosa"})
store.put(namespace, "north_indian_receipes", {"text": "I like paratha, chhole"})

# get data rom store
item = store.get(namespace, "south_indian_receipes")
print(f'Direct get ', item.value)

print('All facts info per namespace')
for result in store.search(namespace):
    print(" - ", result.value["text"])


Direct get  {'text': 'I like idli, wada, dosa'}
All facts info per namespace
 -  I like idli, wada, dosa
 -  I like paratha, chhole


In [17]:
# Approach 2 - Memory Tools: Let agent manage its own memory
from langgraph.config import get_store, get_config

@tool
def save_memory(fact: str) -> str:
    """Saves an important fact about the user for future conversation"""
    store = get_store()
    user_id = get_config()["configurable"]["user_id"]
    store.put(("memories", user_id), fact[:40], {"text": fact})
    return f"Saved: {fact}"

@tool
def recall_memory(query: str) -> str:
    """Search saved facts about the user relevant to the query.
    Always call this before claiming you don't know something about the user.
    """
    store = get_store()
    user_id = get_config()["configurable"]["user_id"]
    results = store.search(("memories", user_id), query=query, limit=5)
    for result in results:
        print(" - ", result.value["text"])
    return results

print('Tools save_memory & recall_memory created...')


Tools save_memory & recall_memory created...


In [18]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

long_term_store = InMemoryStore()
short_term_checkpointer = MemorySaver()

agent_with_store = create_agent(
    model=llm,
    tools=[get_weather, get_stock_price, save_memory, recall_memory],
    system_prompt=(
        "You are a helpful assistant with long term memory."
        "When the user shares the personal fact, call save_memory."
        "When asked what you know about that user, call recall_memory"
    ),
    checkpointer=short_term_checkpointer,
    store=long_term_store
)
print('Agent with short term & long term memries created...')

Agent with short term & long term memries created...


In [19]:
config = {"configurable": {"thread_id": "South_indian_dishes_receipes", "user_id": "user_id_tom"}}

print('Turn 1')
response_1 = agent_with_store.invoke(
    {"messages": [{"role": "user", "content": "My name is Anand and I work with Zensar Technologies Pune"}]},
    config=config
)
print(f'Agent: {response_1['messages'][-1].content}\n')

print('Turn 2')
response_3 = agent_with_store.invoke(
    {"messages": [{"role": "user", "content": "What is my name and where do I work?"}]},
    config=config
)
print(f'Agent: {response_3['messages'][-1].content}\n')

Turn 1
Agent: Nice to meet you, Anand! 🎉 I’ve noted that you work with Zensar Technologies in Pune. How can I assist you today?

Turn 2
 -  User's name is Anand and they work with Zensar Technologies Pune.
Agent: Your name is **Anand**, and you work with **Zensar Technologies in Pune**.



In [22]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="nomic-embed-text")

semantic_store = InMemoryStore(index={"embed": embeddings, "dims": None, "fields": ["text"]})

namespace = ("memories", "user_id_tom")
semantic_store.put(namespace, "fact1", {"text": "My name is Anand"})
semantic_store.put(namespace, "fact2", {"text": "I work with Zensar"})
semantic_store.put(namespace, "fact3", {"text": "I like music"})

results = semantic_store.search(namespace, query="What is my name?")
for result in results:
    print(f" score={result.score} -> {result.value["text"]}")

 score=0.578749122946072 -> My name is Anand
 score=0.38866655492757696 -> I like music
 score=0.3489158777163486 -> I work with Zensar


In [ ]:
# Approach 4 - Sqlite for long term memory
from langgraph.store.sqlite import SqliteStore

db_path = "long_term_memory.db"

with SqliteStore.from_conn_string(db_path) as persistent_store:
    persistent_store.setup()  # creates tables on first run

    persistent_store.put(("memories", "anand_2001"), "job", {"text": "Works at Zensar Technologies"})
    persistent_store.put(("memories", "anand_2001"), "unit_pref", {"text": "Prefers Celsius"})

    print("Saved to disk. Simulating an app restart now...")


